In [1]:
"""
MLS/Aura HNO3 Downloader  —  NASA GES DISC
============================================
Downloads EOS Aura Microwave Limb Sounder (MLS) Level 2 nitric acid (HNO3)
swath files from the NASA GES DISC HTTPS archive.

Product : ML2HNO3 v006  (ML2HNO3.006)
Variable: HNO3_L2gpValue  [nProfs × nPressLevels]
          useful range 215–1.47 hPa (stratospheric + upper-trop)
Sampling: ~3500 profiles/day, ~15 orbits/day, 1.5° spacing along-track
Coverage: 2004-08-08 → present  (−82° to +82° latitude)
Format  : HDF-EOS5 (.he5), typically 30–80 MB per day

Output:
    D:\\Satellites\\MLS\\ML2HNO3\\{year}\\   ← one .he5 per day

URL pattern (HTTPS direct):
    https://acdisc.gesdisc.eosdis.nasa.gov/data/Aura_MLS_Level2/ML2HNO3.006/{year}/{doy:03d}/

Granule discovery:
    NASA CMR search API — returns exact filenames without hardcoding
    version strings that change between processing runs.

Auth:
    Earthdata Bearer token (same token used for LAADS / ASDC).
    The token must belong to a URS account that has approved the
    "NASA GESDISC DATA ARCHIVE" application.

HDF-EOS5 read snippet (for reference, not used during download):
    import h5py, numpy as np
    with h5py.File("MLS-Aura_L2GP-HNO3_...he5", "r") as f:
        hno3  = f["HDFEOS/SWATHS/HNO3/Data Fields/L2gpValue"][:]   # [nP, nLev]
        pres  = f["HDFEOS/SWATHS/HNO3/Geolocation Fields/Pressure"][:]
        lat   = f["HDFEOS/SWATHS/HNO3/Geolocation Fields/Latitude"][:]
        lon   = f["HDFEOS/SWATHS/HNO3/Geolocation Fields/Longitude"][:]
        # valid range mask: Status==0, Quality>0.2, Convergence<2.0
        # See MLS L2 v6 Data Quality document for full screening rules.

References:
    Manney et al. (2026), doi:10.5067/AURA/MLS/DATA2611
    MLS v6 L2 Data Quality Document (GES DISC)
"""

import re
import time
import requests
from pathlib import Path
from datetime import date, timedelta

# ── Earthdata token ────────────────────────────────────────────────────────────
# Same token as used for LAADS DAAC / ASDC CALIPSO downloads.
TOKEN = (
    "eyJ0eXAiOiJKV1QiLCJvcmlnaW4iOiJFYXJ0aGRhdGEgTG9naW4iLCJzaWciOiJlZGxqd3RwdWJrZXlfb3BzIiwiYWxnIjoiUlMyNTYifQ"
    ".eyJ0eXBlIjoiVXNlciIsInVpZCI6ImVtbWFudWVsX2NoZXZhc3N1czEiLCJleHAiOjE3ODMzMzA1MTcsImlhdCI6MTc3ODE0NjUxNywiaXNzIjoiaHR0cHM6Ly91cnMuZWFydGhkYXRhLm5hc2EuZ292IiwiaWRlbnRpdHlfcHJvdmlkZXIiOiJlZGxfb3BzIiwiYWNyIjoiZWRsIiwiYXNzdXJhbmNlX2xldmVsIjozfQ"
    ".WrfjLCrIqdU3hYWV_Z0qHFCMX7UU_J60nY9vTO_PszHYYFLfkGxIja4XzFCA-TMgFraqxujK_G4tIbC7B0fM83QAqEKBA0X9hZEGbBjNNus0iDJnRHA1UqX6M4RKLpPXfNF6iRhztPzuK5s2-2E5oSXpWJyJyiNeJPXPmzda2AmnoJsidmBaOGibyhB6aoJrn-tGqGm470Yozl5vL9M3UDfmyNCmBWi-BOLhxWtzJeRNF9w2GMEDIpi6Dz55sDn4NS9lhE1bmD6Os8emUoB8y9PbvpklaRd6pDQP64wNb12ol_G56ot4bc1pw3dpQpUCSoido5pZWZUaBOOky2grkA"
)

# ── Configuration ──────────────────────────────────────────────────────────────
# MLS/Aura first useful HNO3 data: 2004-08-08
START_DATE = date(2004, 8, 8)
END_DATE   = date(2025, 12, 31)   # set to date.today() to run to present

OUTPUT_ROOT = Path(r"D:\Satellites\MLS\ML2HNO3")

# GES DISC constants
GES_DISC_BASE = "https://acdisc.gesdisc.eosdis.nasa.gov/data/Aura_MLS_Level2/ML2HNO3.006"
CMR_SEARCH    = "https://cmr.earthdata.nasa.gov/search/granules.json"
SHORT_NAME    = "ML2HNO3"
VERSION       = "006"

REQUEST_DELAY = 0.3   # seconds between file downloads (GES DISC is not rate-limited
                       # like LAADS, but be polite)
CMR_PAGE_SIZE = 100   # granules per CMR page (max 2000, but 100 is safe)

# ── Session ────────────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "User-Agent":    "python-requests/AEROTREND-MLS-HNO3",
})


# ── CMR granule search ─────────────────────────────────────────────────────────

def cmr_granules_for_day(d: date) -> list[tuple[str, str]]:
    """
    Query NASA CMR for all ML2HNO3 v006 granules on date `d`.
    Returns list of (filename, https_url).
    GES DISC typically produces exactly ONE granule per day for MLS L2.
    """
    t0 = f"{d.isoformat()}T00:00:00Z"
    t1 = f"{d.isoformat()}T23:59:59Z"

    params = {
        "short_name":        SHORT_NAME,
        "version":           VERSION,
        "temporal":          f"{t0},{t1}",
        "page_size":         CMR_PAGE_SIZE,
        "sort_key":          "start_date",
    }

    try:
        r = session.get(CMR_SEARCH, params=params, timeout=30)
    except requests.RequestException as e:
        print(f"    [CMR network error] {d}: {e}")
        return []

    if r.status_code != 200:
        print(f"    [CMR HTTP {r.status_code}] {d}")
        return []

    entries = r.json().get("feed", {}).get("entry", [])
    results = []

    for entry in entries:
        # Walk the links array for the HTTPS data link (not OPeNDAP, not browse)
        for link in entry.get("links", []):
            href = link.get("href", "")
            rel  = link.get("rel", "")
            # GES DISC data links end in .he5 and have rel ending in 'data#'
            if href.endswith(".he5") and ("data#" in rel or "download" in rel.lower()):
                filename = href.rsplit("/", 1)[-1]
                results.append((filename, href))
                break
        else:
            # Fallback: reconstruct URL from producer_granule_id
            pgid = entry.get("producer_granule_id", "")
            if pgid.endswith(".he5"):
                doy = d.timetuple().tm_yday
                url = f"{GES_DISC_BASE}/{d.year}/{doy:03d}/{pgid}"
                results.append((pgid, url))

    return results


# ── File download ──────────────────────────────────────────────────────────────

def download_file(url: str, outdir: Path, filename: str) -> bool:
    """Download one .he5 file. Skip if already present and non-zero."""
    outfile = outdir / filename
    if outfile.exists() and outfile.stat().st_size > 0:
        return True   # already have it

    tmp = outfile.with_suffix(".tmp")
    try:
        r = session.get(url, timeout=600, stream=True, allow_redirects=True)
    except requests.RequestException as e:
        print(f"      [network error] {filename}: {e}")
        return False

    if r.status_code == 401:
        print(f"      [401 Unauthorized] {filename} — check token / GES DISC app approval")
        return False
    if r.status_code == 404:
        print(f"      [404 Not Found] {filename}")
        return False
    if r.status_code != 200:
        print(f"      [HTTP {r.status_code}] {filename}")
        return False

    ct = r.headers.get("Content-Type", "")
    if "text/html" in ct:
        print(f"      [got HTML — auth redirect?] {filename}")
        return False

    try:
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=4 << 20):   # 4 MB chunks
                f.write(chunk)
        tmp.rename(outfile)
    except OSError as e:
        print(f"      [write error] {filename}: {e}")
        if tmp.exists():
            tmp.unlink()
        return False

    size_mb = outfile.stat().st_size / 1e6
    print(f"      + {filename}  ({size_mb:.0f} MB)")
    return True


# ── Date helpers ───────────────────────────────────────────────────────────────

def dates_in_range(start: date, end: date):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)


def year_outdir(year: int) -> Path:
    """Return (and create) output directory for a given year."""
    p = OUTPUT_ROOT / str(year)
    p.mkdir(parents=True, exist_ok=True)
    return p


# ── Auth probe ─────────────────────────────────────────────────────────────────

def probe_auth() -> bool:
    """
    Try to HEAD a known ML2HNO3 v006 file to confirm auth is working.
    Uses CMR to find a real URL for the first available day.
    """
    test_day = date(2004, 8, 8)
    granules = cmr_granules_for_day(test_day)
    if not granules:
        print("  WARNING: CMR returned no granules for 2004-08-08 — cannot verify auth.")
        return False

    _, url = granules[0]
    try:
        r = session.head(url, timeout=20, allow_redirects=True)
        if r.status_code == 200:
            print(f"  Auth OK — GES DISC reachable  ({r.status_code})")
            print(f"  Sample URL: {url}")
            return True
        elif r.status_code == 401:
            print("  AUTH FAILED (401). Ensure your Earthdata account has approved")
            print("  the 'NASA GESDISC DATA ARCHIVE' application at:")
            print("  https://urs.earthdata.nasa.gov/approve_app?client_id=e2WVk8Poinhu6fcdinaEEQ")
            return False
        else:
            print(f"  Auth probe returned HTTP {r.status_code} — proceeding anyway.")
            return True
    except requests.RequestException as e:
        print(f"  Auth probe failed: {e}")
        return False


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    print("MLS/Aura HNO3 Downloader  (GES DISC  ML2HNO3 v006)")
    print(f"  Date range : {START_DATE} → {END_DATE}")
    print(f"  Output     : {OUTPUT_ROOT}")
    print()

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    # Verify auth before starting the loop
    if not probe_auth():
        print("\nAborting — fix auth before running.")
        return
    print()

    total_ok = total_skip = total_fail = total_missing = 0

    for d in dates_in_range(START_DATE, END_DATE):

        outdir = year_outdir(d.year)

        # ── CMR lookup ────────────────────────────────────────────────────────
        granules = cmr_granules_for_day(d)

        if not granules:
            # MLS had brief data gaps (instrument issues); skip silently.
            total_missing += 1
            continue

        for filename, url in granules:
            outfile = outdir / filename
            if outfile.exists() and outfile.stat().st_size > 0:
                total_skip += 1
                continue

            print(f"  {d.isoformat()}  →  {filename}")
            ok = download_file(url, outdir, filename)
            if ok:
                total_ok += 1
            else:
                total_fail += 1
            time.sleep(REQUEST_DELAY)

    print()
    print("=" * 60)
    print(f"  Downloaded  : {total_ok}")
    print(f"  Skipped     : {total_skip}  (already present)")
    print(f"  Failed      : {total_fail}")
    print(f"  Missing days: {total_missing}  (no CMR granule — instrument gaps)")
    print("All done.")


if __name__ == "__main__":
    main()

MLS/Aura HNO3 Downloader  (GES DISC  ML2HNO3 v006)
  Date range : 2004-08-08 → 2025-12-31
  Output     : D:\Satellites\MLS\ML2HNO3

  Auth OK — GES DISC reachable  (200)
  Sample URL: https://data.gesdisc.earthdata.nasa.gov/data/Aura_MLS_Level2/ML2HNO3.006/2004/MLS-Aura_L2GP-HNO3_v06-01-c01_2004d221.he5

  2004-08-08  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d221.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2004d221.he5  (4 MB)
  2004-08-09  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d222.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2004d222.he5  (3 MB)
  2004-08-10  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d223.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2004d223.he5  (3 MB)
  2004-08-11  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d224.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2004d224.he5  (2 MB)
  2004-08-12  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d225.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2004d225.he5  (4 MB)
  2004-08-13  →  MLS-Aura_L2GP-HNO3_v06-01-c01_2004d226.he5
      + MLS-Aura_L2GP-HNO3_v06-01-c01_2